<span style="color:green"> Import every csv file into their respective table in the database.</span>


In [1]:
import os
import tempfile

import pandas as pd
import sqlalchemy as sqla

TRAIN_DATA_COLUMNS = [
    "id",
    "departure_date",
    "train_number",
    "relation",
    "train_service",
    "ptcar_number",
    "line_number_departure",
    "real_time_arrival",
    "real_time_departure",
    "planned_time_arrival",
    "planned_time_departure",
    "delay_arrival",
    "delay_departure",
    "ptcar_name",
    "line_number_arrival",
    "station_departure",
    "station_arrival",
]


def load_csv_table(engine, file_path, table_name, columns):
    connection = engine.raw_connection()
    try:
        with connection.cursor() as cursor:
            cursor.execute(f"SELECT EXISTS (SELECT 1 FROM {table_name})")
            already_loaded = cursor.fetchone()[0]
            if not already_loaded:
                column_list = ", ".join(columns)
                with open(file_path, encoding="utf-8", newline="") as handle:
                    cursor.copy_expert(
                        f"COPY {table_name} ({column_list}) FROM STDIN "
                        "WITH CSV HEADER DELIMITER ',' NULL ''",
                        handle,
                    )
            if table_name == "stations":
                cursor.execute(
                    "UPDATE stations SET geom = "
                    "ST_SetSRID(ST_MakePoint(longitude, latitude), 4326)"
                )
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()
    status = "already loaded" if already_loaded else "loaded"
    print(f"{table_name}: {status}")


def load_monthly_csv(engine, file_path):
    filename = os.path.basename(file_path)
    connection = engine.raw_connection()
    try:
        with connection.cursor() as cursor:
            # Serialize imports so explicit stop ids cannot overlap.
            cursor.execute("LOCK TABLE train_data IN SHARE ROW EXCLUSIVE MODE")
            cursor.execute(
                "SELECT 1 FROM imported_train_files WHERE file_name = %s",
                (filename,),
            )
            if cursor.fetchone() is not None:
                print(f"{filename}: already loaded")
                return
            cursor.execute("SELECT COALESCE(MAX(id), 0) FROM train_data")
            last_id = cursor.fetchone()[0]
            cursor.execute(
                "SELECT EXISTS (SELECT 1 FROM imported_train_files)"
            )
            if last_id and not cursor.fetchone()[0]:
                raise RuntimeError(
                    "Existing train_data has no import checkpoints. "
                    "Migrate its progress or use a new database "
                    "before loading."
                )
            integer_columns = [
                "train_number",
                "relation",
                "train_service",
                "ptcar_number",
                "ptcar_name",
                "station_departure",
                "station_arrival",
            ]
            monthly_data = pd.read_csv(
                file_path,
                dtype={column: "Int64" for column in integer_columns},
            )
            row_count = len(monthly_data)
            monthly_data.insert(
                0, "id", range(last_id + 1, last_id + row_count + 1)
            )
            with tempfile.TemporaryFile(
                mode="w+",
                encoding="utf-8",
                newline="",
                dir=os.path.dirname(file_path),
            ) as temporary_csv:
                monthly_data[TRAIN_DATA_COLUMNS].to_csv(
                    temporary_csv, index=False
                )
                temporary_csv.seek(0)
                column_list = ", ".join(TRAIN_DATA_COLUMNS)
                cursor.copy_expert(
                    f"COPY train_data ({column_list}) FROM STDIN "
                    "WITH CSV HEADER DELIMITER ',' NULL ''",
                    temporary_csv,
                )
            new_last_id = last_id + row_count
            cursor.execute(
                "INSERT INTO imported_train_files "
                "(file_name, row_count, last_id) VALUES (%s, %s, %s)",
                (filename, row_count, new_last_id),
            )
            cursor.execute(
                "SELECT setval(pg_get_serial_sequence('train_data', 'id'), "
                "%s, %s)",
                (new_last_id or 1, new_last_id > 0),
            )
        connection.commit()
    except Exception:
        connection.rollback()
        raise
    finally:
        connection.close()
    print(f"{filename}: loaded {row_count} rows")

In [2]:
DB_NAME = "postgres"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"

SQL_INIT = "../db/init.sql"

engine = sqla.create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Database initialization

In [ ]:
with open(SQL_INIT, encoding="utf-8") as handle:
    schema = handle.read()
with engine.begin() as connection:
    connection.execute(sqla.text(schema))
print("Database initialized")

(psycopg2.errors.DuplicateTable) relation "service" already exists

[SQL: START TRANSACTION;

-- For TRAIN_SERV
CREATE TABLE SERVICE (
    id INT PRIMARY KEY, -- id of the service, autoincrement
    name VARCHAR(100) NOT NULL -- name of the service
) ;

CREATE TABLE REGION(
    id INT PRIMARY KEY, -- id of the region, autoincrement
    name VARCHAR(255) NOT NULL -- name of the region
) ;

-- For UNIQUE_STATION
CREATE TABLE STATIONS (
    id INT PRIMARY KEY, -- id of the station, autoincrement
    name VARCHAR(255) NOT NULL, -- name of the station
    region INT NOT NULL, -- region of the station
    latitude DOUBLE PRECISION NOT NULL, -- latitude of the station
    longitude DOUBLE PRECISION NOT NULL, -- longitude of the station

    FOREIGN KEY (region) REFERENCES REGION(id)
) ;

CREATE TABLE RELATION(
    id INT PRIMARY KEY, -- id of the relation, autoincrement
    name VARCHAR(255) NOT NULL, -- name of the relation
    type INT NOT NULL -- type of the relation
);


CREATE TABLE TRAI

: 

## Region

In [4]:
load_csv_table(
    engine,
    "../data/reference/REGION.csv",
    "region",
    ["id", "name"],
)

REGION loaded


## Service

In [5]:
load_csv_table(
    engine,
    "../data/reference/TRAIN_SERV.csv",
    "service",
    ["id", "name"],
)

Service loaded


## Stations

In [6]:
load_csv_table(
    engine,
    "../data/reference/stations_with_regions.csv",
    "stations",
    ["id", "name", "region", "latitude", "longitude"],
)

Stations loaded


## RELATION

In [7]:
load_csv_table(
    engine,
    "../data/reference/RELATION.csv",
    "relation",
    ["id", "name", "type"],
)

RELATION loaded


## Dataset

In [ ]:
DATASET_DIR = "../data/cleaned"
DATASET_FILES = sorted(
    filename
    for filename in os.listdir(DATASET_DIR)
    if filename.endswith(".csv") and not filename.startswith("temp_")
)

for filename in DATASET_FILES:
    load_monthly_csv(engine, os.path.join(DATASET_DIR, filename))

Data_raw_punctuality_201401.csv loaded successfully
Data_raw_punctuality_201402.csv loaded successfully
Data_raw_punctuality_201403.csv loaded successfully
Data_raw_punctuality_201404.csv loaded successfully
Data_raw_punctuality_201405.csv loaded successfully
Data_raw_punctuality_201406.csv loaded successfully
Data_raw_punctuality_201407.csv loaded successfully
Data_raw_punctuality_201408.csv loaded successfully
Data_raw_punctuality_201409.csv loaded successfully
Data_raw_punctuality_201410.csv loaded successfully
Data_raw_punctuality_201411.csv loaded successfully
Data_raw_punctuality_201412.csv loaded successfully
Data_raw_punctuality_201501.csv loaded successfully
Data_raw_punctuality_201502.csv loaded successfully
Data_raw_punctuality_201503.csv loaded successfully
Data_raw_punctuality_201504.csv loaded successfully
Data_raw_punctuality_201505.csv loaded successfully
Data_raw_punctuality_201506.csv loaded successfully
Data_raw_punctuality_201507.csv loaded successfully
Data_raw_pun

: 

# DATE

In [3]:
load_csv_table(
    engine,
    "../data/reference/dates_2014_2024.csv",
    "type_day",
    ["date", "holiday", "weekend", "day_after_rest"],
)

type_day loaded


## PROVINCE

In [4]:
load_csv_table(
    engine,
    "../data/reference/province_mapping.csv",
    "province",
    ["id", "province", "region"],
)

province loaded


## WEATHER

In [4]:
load_csv_table(
    engine,
    "../data/weather/hourly_weather_data.csv",
    "weather",
    [
        "temperature",
        "dewpoint",
        "relative_humidity",
        "precipitation",
        "snowfall",
        "wind_direction",
        "wind_speed",
        "pressure",
        "province",
        "date",
        "hour",
    ],
)

weather loaded
